# 12. Modelo 2 – ETS (Error, Trend, Seasonality)

El modelo ETS utiliza suavización exponencial para capturar patrones. Dado que los datos son anuales sin estacionalidad clara, se aplica suavización con parámetro α optimizado automáticamente.

- **Fortaleza:** Simple, robusto y fácil de interpretar
- **Limitación:** No captura cambios abruptos en la tendencia

## 12.1. Diagnóstico de residuos – ETS

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox

Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
wv_acc = (estados[(estados['state']=='West Virginia')&
                    (estados['cause_name']=='Unintentional injuries')]
           .sort_values('year').dropna(subset=['age_adjusted_death_rate']))
serie = wv_acc.set_index('year')['age_adjusted_death_rate']

modelo_ets = ExponentialSmoothing(serie, trend='add', seasonal=None).fit(optimized=True)
print("=== Modelo ETS ===")
print(f"Alpha (nivel): {modelo_ets.params['smoothing_level']:.4f}")
print(f"Beta  (trend): {modelo_ets.params.get('smoothing_trend',0):.4f}")
print(f"AIC:           {modelo_ets.aic:.2f}")
print(f"BIC:           {modelo_ets.bic:.2f}")

resid_ets = modelo_ets.resid
lb_ets = acorr_ljungbox(resid_ets, lags=[4], return_df=True)
print(f"\nLjung-Box Q*: {lb_ets['lb_stat'].values[0]:.4f}, p={lb_ets['lb_pvalue'].values[0]:.4f}")
print(f"{'✔ Residuos independientes' if lb_ets['lb_pvalue'].values[0]>0.05 else '⚠ Autocorrelación'}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(serie.index), y=resid_ets.values, mode='lines+markers',
                          line=dict(color='#27ae60'), name='Residuos ETS'))
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(title='Residuos – Modelo ETS', height=340, template='plotly_white')
fig.show()

=== Modelo ETS ===
Alpha (nivel): 0.0000
Beta  (trend): 0.0000
AIC:           79.47
BIC:           83.24

Ljung-Box Q*: 4.7703, p=0.3117
✔ Residuos independientes


## 12.2. Proyección ETS (2018–2022)

In [3]:
anios_fut = list(range(2018, 2023))
fc_ets = modelo_ets.forecast(5)
pred_ets = modelo_ets.predict(start=serie.index[0], end=serie.index[-1])

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(serie.index), y=serie.values, mode='lines+markers',
                          name='Observado', line=dict(color='#1D3557', width=2)))
fig.add_trace(go.Scatter(x=list(serie.index), y=pred_ets.values, mode='lines',
                          name='Ajustado ETS', line=dict(color='#27ae60', width=1.5)))
fig.add_trace(go.Scatter(x=anios_fut, y=fc_ets.values, mode='lines+markers',
                          name='Proyección ETS', line=dict(color='#e74c3c', width=2, dash='dash'),
                          marker=dict(size=8, symbol='triangle-up')))
fig.add_vline(x=2017.5, line_dash='dot', line_color='gray')
fig.update_layout(title='Proyección ETS – Unintentional Injuries · West Virginia (2018–2022)',
                   xaxis_title='Año', yaxis_title='Tasa por 100,000 hab.',
                   height=450, template='plotly_white')
fig.show()

## 12.3. Tabla de valores proyectados

In [4]:
tabla_ets = pd.DataFrame({'Año':anios_fut, 'Predicción ETS':fc_ets.values.round(1)})
print(tabla_ets.to_string(index=False))

 Año  Predicción ETS
2018            88.8
2019            91.4
2020            93.9
2021            96.4
2022            99.0
